# 04 - Metrics and the scale sweep

Two metrics survived, one did not.

**Edge F1** (structural fidelity) - re-extract canny from the output and compare it with the input's, inside the product region only. Stage 2 scores 0.90, stage 3 drops to 0.82: IC-Light rewrites shading boundaries, and canny counts those as edges.

**CLIP score** (prompt adherence) - stage 3 scores higher on both the scene prompt (+3.4%) and the light prompt (+6.4%). The larger relative gain on the light prompt is weak but consistent evidence that CLIP sees the lighting change; it is not a sensitive instrument for light direction.

**Lighting consistency - dropped.** Two formulations were tried. Product/background histogram agreement turned out to reward *flat* lighting: stage 3 scored worse than stage 2 precisely because it added dramatic shadows. Gradient-direction agreement was dominated by silhouette contrast and returned nearly the same value regardless of input. Documented as a limitation instead of reported as a number.

**Scale sweep result:** edge F1 rises from 0.750 (no ControlNet) to 0.899 at scale 0.6, then plateaus. CLIP score rises slightly throughout, because the canny map is cropped to the product and the background is never constrained. Selected working value: **0.6**.

In [ ]:
import os, sys
sys.path.insert(0, "../src")
os.chdir("..")
print("working dir:", os.getcwd())

## 1. Stage 2 vs stage 3

In [ ]:
import numpy as np
from PIL import Image
from metrics import edge_f1, clip_score

PROMPT = ("professional product photo of a white ceramic mug on a marble "
          "kitchen counter, soft studio lighting, high quality")
LIGHT_PROMPT = "warm golden hour light from left side, soft shadows"

source = Image.open("inputs/testset/01_matte_mug.jpeg").resize((512, 640))
mask = Image.open("outputs/mask_rembg.png")

for name in ["e2e_stage2", "e2e_final"]:
    image = Image.open(f"outputs/{name}.png")
    scores = edge_f1(source, image, mask)
    print(f"{name:12s} F1={scores['f1']:.4f}  P={scores['precision']:.4f}  "
          f"R={scores['recall']:.4f}  CLIP={clip_score(image, PROMPT):.4f}  "
          f"CLIP-light={clip_score(image, LIGHT_PROMPT):.4f}")

Precision is below recall in both stages: the pipeline invents more edges than it erases, which is the general form of the gold-rim artifact from notebook 01.

## 2. Scale sweep

Only the inpainting stage is swept, one variable at a time. Scale 0.0 is the no-ControlNet baseline - without it the other numbers have nothing to be compared against.

In [ ]:
import gc
import torch
from pipeline import load_models, build_masks, build_canny, SIZE, NEGATIVE_PROMPT

models = load_models()
inpaint_mask, product_mask = build_masks(source.convert("RGB"), models["rembg"])
control_image = build_canny(source.convert("RGB"), product_mask)
mask_pil = Image.fromarray(product_mask)

results = []
for scale in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2]:
    generator = torch.Generator(device="cuda").manual_seed(42)
    out = models["pipe"](
        prompt=PROMPT, negative_prompt=NEGATIVE_PROMPT,
        image=source.convert("RGB"), mask_image=inpaint_mask,
        control_image=control_image, num_inference_steps=25,
        controlnet_conditioning_scale=scale, generator=generator,
        height=SIZE[1], width=SIZE[0]).images[0]
    out.save(f"outputs/sweep_scale_{scale}.png")

    f1 = edge_f1(source, out, mask_pil)["f1"]
    clip = clip_score(out, PROMPT)
    results.append((scale, f1, clip))
    print(f"scale={scale:.1f}  edge_F1={f1:.4f}  CLIP={clip:.4f}")

del models
gc.collect()
torch.cuda.empty_cache()

## 3. Trade-off plot

In [ ]:
import matplotlib.pyplot as plt

scales, f1_values, clip_values = zip(*results)
fig, ax1 = plt.subplots(figsize=(8, 5))

ax1.plot(scales, f1_values, "o-", color="#2E5395", lw=2, label="Edge F1 (structural fidelity)")
ax1.set_xlabel("controlnet_conditioning_scale")
ax1.set_ylabel("Edge F1", color="#2E5395")
ax1.tick_params(axis="y", labelcolor="#2E5395")
ax1.grid(alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(scales, clip_values, "s--", color="#C0504D", lw=2, label="CLIP score (prompt adherence)")
ax2.set_ylabel("CLIP score", color="#C0504D")
ax2.tick_params(axis="y", labelcolor="#C0504D")

handles = ax1.get_legend_handles_labels()[0] + ax2.get_legend_handles_labels()[0]
labels = ax1.get_legend_handles_labels()[1] + ax2.get_legend_handles_labels()[1]
ax1.legend(handles, labels, loc="lower right")

plt.title("Fidelity vs prompt adherence in the inpainting stage (seed 42)")
plt.tight_layout()
plt.savefig("assets/scale_sweep.png", dpi=150)
plt.show()

**Caveat:** the sweep uses a single seed. The small bump in CLIP score at 0.2 is most likely seed noise; a rigorous version would average 3-5 seeds per point.